In [ ]:
# Import library numerik
# Untuk komputasi numerik seperti log, array, differencing
import numpy as np

# Import library untuk manipulasi dan analisis data dalam bentuk tabel
# Untuk membaca dan mengolah data dalam bentuk DataFrame
import pandas as pd

# Import fungsi ADF (Augmented Dickey-Fuller) Test dari statsmodels untuk uji stasioneritas data time series
# Untuk menguji apakah data time series stasioner
from statsmodels.tsa.stattools import adfuller

# Import matplotlib untuk visualisasi data seperti grafik tren dan forecast
# Untuk membuat grafik
import matplotlib.pyplot as plt

# Import ARIMA model dari statsmodels untuk forecasting data time series yang mengandung tren
# Untuk membangun dan melatih model ARIMA
from statsmodels.tsa.arima.model import ARIMA

# Import Exponential Smoothing (ETS) model dari statsmodels untuk data dengan tren dan musiman
# Untuk model ETS (trend dan seasonality)
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Import MAPE dari sklearn untuk mengevaluasi akurasi model forecasting
# Untuk menghitung error model (semakin kecil semakin baik)
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

# Import joblib untuk menyimpan dan memuat model yang sudah dilatih
import joblib  # Untuk save/load model ke file .pkl (agar tidak perlu dilatih ulang)

In [ ]:
df_1 = pd.read_csv('/content/sales_data_january_2019.csv')
df_2 = pd.read_csv('/content/sales_data_february_2019 (1).csv')
df_3 = pd.read_csv('/content/sales_data_march_2019 (1).csv')
df_4 = pd.read_csv('/content/sales_data_april_2019.csv')
df_5 = pd.read_csv('/content/sales_data_may_2019.csv')
df_6 = pd.read_csv('/content/sales_data_june_2019.csv')
df_7 = pd.read_csv('/content/sales_data_july_2019.csv')
df_8 = pd.read_csv('/content/sales_data_august_2019.csv')
df_9 = pd.read_csv('/content/sales_data_september_2019.csv')
df_10 = pd.read_csv('/content/sales_data_october_2019.csv')
df_11 = pd.read_csv('/content/sales_data_november_2019.csv')
df_12 = pd.read_csv('/content/sales_data_december_2019.csv')

In [ ]:
df = pd.concat([df_1, df_2 , df_3 , df_4 , df_5 , df_6 , df_7 , df_8 , df_9 , df_10 , df_11 , df_12], ignore_index=True)
df

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
0,141234,iPhone,1,700,01/22/19 21:25,"944 Walnut St, Boston, MA 02215"
1,141235,Lightning Charging Cable,1,14.95,01/28/19 14:15,"185 Maple St, Portland, OR 97035"
2,141236,Wired Headphones,2,11.99,01/17/19 13:33,"538 Adams St, San Francisco, CA 94016"
3,141237,27in FHD Monitor,1,149.99,01/05/19 20:33,"738 10th St, Los Angeles, CA 90001"
4,141238,Wired Headphones,1,11.99,01/25/19 11:59,"387 10th St, Austin, TX 73301"
...,...,...,...,...,...,...
186845,319666,Lightning Charging Cable,1,14.95,12/11/19 20:58,"14 Madison St, San Francisco, CA 94016"
186846,319667,AA Batteries (4-pack),2,3.84,12/01/19 12:01,"549 Willow St, Los Angeles, CA 90001"
186847,319668,Vareebadd Phone,1,400,12/09/19 06:43,"273 Wilson St, Seattle, WA 98101"
186848,319669,Wired Headphones,1,11.99,12/03/19 10:39,"778 River St, Dallas, TX 75001"


# Data Cleansing

In [ ]:
df.isna().sum()

,0
Order ID,545
Product,545
Quantity Ordered,545
Price Each,545
Order Date,545
Purchase Address,545


In [ ]:
df.isna().sum()/len(df)

,0
Order ID,0.002917
Product,0.002917
Quantity Ordered,0.002917
Price Each,0.002917
Order Date,0.002917
Purchase Address,0.002917


# Handling Data Cleansing

In [ ]:
df = df.dropna()
df.isna().sum()

,0
Order ID,0
Product,0
Quantity Ordered,0
Price Each,0
Order Date,0
Purchase Address,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 186305 entries, 0 to 186849
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Order ID          186305 non-null  object
 1   Product           186305 non-null  object
 2   Quantity Ordered  186305 non-null  object
 3   Price Each        186305 non-null  object
 4   Order Date        186305 non-null  object
 5   Purchase Address  186305 non-null  object
dtypes: object(6)
memory usage: 9.9+ MB


# Cek Duplikasi

In [ ]:
df.duplicated().sum()

np.int64(618)

In [ ]:
df[df.duplicated()]

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
875,142071,AA Batteries (4-pack),1,3.84,01/17/19 23:02,"131 2nd St, Boston, MA 02215"
1102,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
1194,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
1897,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
2463,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
...,...,...,...,...,...,...
185070,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
185085,317971,AA Batteries (4-pack),1,3.84,12/17/19 18:39,"250 Chestnut St, San Francisco, CA 94016"
185481,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
185925,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address


In [ ]:
df = df[df['Order ID'] != 'Order ID']
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')

/tmp/ipython-input-1674083094.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')


In [ ]:
df_clean = df.drop_duplicates(
    subset=['Order ID', 'Product', 'Order Date'],
    keep='first')
df[df.duplicated()]

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
875,142071,AA Batteries (4-pack),1,3.84,2019-01-17 23:02:00,"131 2nd St, Boston, MA 02215"
4126,145143,Lightning Charging Cable,1,14.95,2019-01-06 03:01:00,"182 Jefferson St, San Francisco, CA 94016"
5811,146765,Google Phone,1,600,2019-01-21 11:23:00,"918 Highland St, New York City, NY 10001"
6807,147707,Wired Headphones,1,11.99,2019-01-04 16:50:00,"883 4th St, Dallas, TX 75001"
8134,148984,USB-C Charging Cable,1,11.95,2019-01-08 17:36:00,"562 14th St, Boston, MA 02215"
...,...,...,...,...,...,...
181627,314675,AA Batteries (4-pack),1,3.84,2019-12-26 09:01:00,"927 13th St, San Francisco, CA 94016"
182185,315204,Wired Headphones,1,11.99,2019-12-12 12:41:00,"680 6th St, San Francisco, CA 94016"
182973,315955,ThinkPad Laptop,1,999.99,2019-12-26 17:28:00,"588 Chestnut St, Seattle, WA 98101"
183200,316173,AAA Batteries (4-pack),1,2.99,2019-12-22 22:44:00,"907 Sunset St, Portland, OR 97035"


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 185950 entries, 0 to 186849
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Order ID          185950 non-null  object        
 1   Product           185950 non-null  object        
 2   Quantity Ordered  185950 non-null  object        
 3   Price Each        185950 non-null  object        
 4   Order Date        185950 non-null  datetime64[ns]
 5   Purchase Address  185950 non-null  object        
dtypes: datetime64[ns](1), object(5)
memory usage: 9.9+ MB


# Data Manipulation

In [ ]:
df_clean['Quantity Ordered'] = pd.to_numeric(df_clean['Quantity Ordered'], errors='coerce')
df_clean['Price Each'] = pd.to_numeric(df_clean['Price Each'], errors='coerce')

/tmp/ipython-input-1244040293.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Quantity Ordered'] = pd.to_numeric(df_clean['Quantity Ordered'], errors='coerce')
/tmp/ipython-input-1244040293.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Price Each'] = pd.to_numeric(df_clean['Price Each'], errors='coerce')


In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 185639 entries, 0 to 186849
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Order ID          185639 non-null  object        
 1   Product           185639 non-null  object        
 2   Quantity Ordered  185639 non-null  int64         
 3   Price Each        185639 non-null  float64       
 4   Order Date        185639 non-null  datetime64[ns]
 5   Purchase Address  185639 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 9.9+ MB


# Feature Engineering

In [ ]:
df_clean['Revenue'] = df_clean['Quantity Ordered'] * df_clean['Price Each']
df_clean

/tmp/ipython-input-682818532.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Revenue'] = df_clean['Quantity Ordered'] * df_clean['Price Each']


,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Revenue
0,141234,iPhone,1,700.00,2019-01-22 21:25:00,"944 Walnut St, Boston, MA 02215",700.00
1,141235,Lightning Charging Cable,1,14.95,2019-01-28 14:15:00,"185 Maple St, Portland, OR 97035",14.95
2,141236,Wired Headphones,2,11.99,2019-01-17 13:33:00,"538 Adams St, San Francisco, CA 94016",23.98
3,141237,27in FHD Monitor,1,149.99,2019-01-05 20:33:00,"738 10th St, Los Angeles, CA 90001",149.99
4,141238,Wired Headphones,1,11.99,2019-01-25 11:59:00,"387 10th St, Austin, TX 73301",11.99
...,...,...,...,...,...,...,...
186845,319666,Lightning Charging Cable,1,14.95,2019-12-11 20:58:00,"14 Madison St, San Francisco, CA 94016",14.95
186846,319667,AA Batteries (4-pack),2,3.84,2019-12-01 12:01:00,"549 Willow St, Los Angeles, CA 90001",7.68
186847,319668,Vareebadd Phone,1,400.00,2019-12-09 06:43:00,"273 Wilson St, Seattle, WA 98101",400.00
186848,319669,Wired Headphones,1,11.99,2019-12-03 10:39:00,"778 River St, Dallas, TX 75001",11.99


In [ ]:
# Setelah yakin datetime, baru ambil tanggal saja (tanpa jam) dan lakukan agregasi
daily_revenue = df_clean.groupby(df_clean['Order Date'].dt.date)[['Revenue','Quantity Ordered']].sum().reset_index()

# Rename kolom agar lebih deskriptif
daily_revenue.columns = ['Date', 'Total_Revenue','Quantity Ordered']

# Konversi kembali kolom 'Date' ke datetime agar bisa dipakai untuk plotting atau time series model
daily_revenue['Date'] = pd.to_datetime(daily_revenue['Date'])

In [ ]:
daily_revenue

,Date,Total_Revenue,Quantity Ordered
0,2019-01-01,65681.94,343
1,2019-01-02,70663.20,367
2,2019-01-03,47046.20,330
3,2019-01-04,62000.22,329
4,2019-01-05,46524.63,355
...,...,...,...
361,2019-12-28,133601.53,928
362,2019-12-29,156005.83,952
363,2019-12-30,151857.82,925
364,2019-12-31,131439.32,884


In [ ]:
ts = daily_revenue.set_index('Date')[['Total_Revenue','Quantity Ordered']]
ts

,Total_Revenue,Quantity Ordered
Date,,
2019-01-01,65681.94,343
2019-01-02,70663.20,367
2019-01-03,47046.20,330
2019-01-04,62000.22,329
2019-01-05,46524.63,355
...,...,...
2019-12-28,133601.53,928
2019-12-29,156005.83,952
2019-12-30,151857.82,925


Uji ADV

In [ ]:
adf_result = adfuller(ts['Total_Revenue'])  # Jalankan ADF Test pada kolom 'Total_Revenue'
print(f'ADF Statistic: {adf_result[0]}')  # Nilai statistik ADF (semakin kecil = lebih stasioner)
print(f'p-value: {adf_result[1]}')      # p-value hasil uji, jika < 0.05 berarti data stasioner

ADF Statistic: -2.353321978103404
p-value: 0.15534645842519762


Nilai p-Value Lebih Besar 0.05 berarti ada perbedaan disini perlu di logaritma transform

In [ ]:
ts_log = np.log(ts)
ts_log_diff = ts_log.diff().dropna()

In [ ]:
adf_result_diff = adfuller(ts_log_diff['Total_Revenue'])
print(f'ADF Statistic after diff: {adf_result_diff[0]}')
print(f'p-value after diff: {adf_result_diff[1]}')

ADF Statistic after diff: -9.418649764690226
p-value after diff: 5.579989935180682e-16


In [ ]:
adf_result = adfuller(ts['Quantity Ordered'])  # Jalankan ADF Test pada kolom 'Total_Revenue'
print(f'ADF Statistic: {adf_result[0]}')  # Nilai statistik ADF (semakin kecil = lebih stasioner)
print(f'p-value: {adf_result[1]}')      # p-value hasil uji, jika < 0.05 berarti data stasioner

ADF Statistic: -2.64694398415637
p-value: 0.08367775198644989


In [ ]:
ts_log = np.log(ts)
ts_log_diff = ts_log.diff().dropna()

In [ ]:
adf_result_diff = adfuller(ts_log_diff['Quantity Ordered'])
print(f'ADF Statistic after diff: {adf_result_diff[0]}')
print(f'p-value after diff: {adf_result_diff[1]}')

ADF Statistic after diff: -12.542024796524315
p-value after diff: 2.303332529756948e-23


Mddel Data SUdah Stasioner

# Model Untuk Revenue

In [ ]:
# Pisahkan data menjadi dua bagian: data latih (train) dan data uji (test)
# Di sini, kita gunakan data Februari untuk melatih model
# Dan data Maret digunakan sebagai validasi untuk menguji performa model

train = ts[:'2019-06-30']
test = ts['2019-07-01':]

In [ ]:
# Buat model ARIMA untuk data time series
# Di sini kita gunakan log(train) agar model lebih stabil terhadap variasi besar (log transform)
# Parameter (1,1,1) artinya: p=1 (Auto REGRESSION), d=1 (differencing), q=1 (MOVING AVERAGE)

model_arima = ARIMA(np.log(train['Total_Revenue']), order=(1,1,1))  # Buat model ARIMA dengan log-transformed data
result_arima = model_arima.fit()  # Latih model ARIMA

# Forecast dalam bentuk log selama panjang data uji (misal Maret = 31 hari)
forecast_arima_log = result_arima.forecast(steps=len(test))

# Kembalikan hasil forecast dari skala log ke skala asli dengan fungsi eksponensial
forecast_arima = np.exp(forecast_arima_log)  # Ini adalah hasil prediksi revenue harian

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


In [ ]:
# Buat model ETS (Exponential Smoothing) dengan komponen tren dan musiman
# 'trend="add"' artinya kita mengasumsikan adanya tren naik/turun yang bersifat aditif
# 'seasonal="add"' artinya ada pola musiman (berulang) yang juga ditambahkan
# 'seasonal_periods=7' menunjukkan adanya pola mingguan (7 hari)

model_ets = ExponentialSmoothing(train['Total_Revenue'], trend='add', seasonal='add', seasonal_periods=7)  # Buat model ETS
result_ets = model_ets.fit()  # Latih modelnya

# Lakukan prediksi sebanyak panjang data uji
forecast_ets = result_ets.forecast(steps=len(test))  # Hasil prediksi revenue harian dari model ETS

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


In [ ]:
# Hitung MAPE (Mean Absolute Percentage Error) untuk mengukur akurasi kedua model
# Semakin kecil nilai MAPE, semakin akurat prediksi model terhadap data aktual

mape_arima = mean_absolute_percentage_error(test['Total_Revenue'], forecast_arima)  # Error ARIMA
mape_ets = mean_absolute_percentage_error(test['Total_Revenue'], forecast_ets)      # Error ETS

# Tampilkan hasil evaluasi dalam format persentase
print(f'MAPE ARIMA: {mape_arima:.2%}')  # Misalnya: 8.45%
print(f'MAPE ETS: {mape_ets:.2%}')

MAPE ARIMA: 28.56%
MAPE ETS: 33.34%


In [ ]:
# Bandingkan nilai MAPE dari ARIMA dan ETS
# Pilih model dengan MAPE terkecil sebagai model terbaik

best_model = "ARIMA" if mape_arima < mape_ets else "ETS"  # Jika MAPE ARIMA lebih kecil, pilih ARIMA, jika tidak pilih ETS

# Tampilkan model terbaik berdasarkan evaluasi MAPE
print(f"Model terbaik berdasarkan MAPE adalah: {best_model}")

Model terbaik berdasarkan MAPE adalah: ARIMA


In [ ]:
# Forecast ulang selama 30 hari (1 bulan ke depan) menggunakan model terbaik yang sudah ditentukan
steps_ahead = 30  # Prediksi 30 hari ke depan

if best_model == "ARIMA":
    # Fit ulang ARIMA ke seluruh data latih
    tuned_arima_model = ARIMA(np.log(train['Total_Revenue']), order=(1,1,1)).fit()

    # Prediksi 30 hari ke depan dalam skala log
    forecast_log = tuned_arima_model.forecast(steps=steps_ahead)

    # Kembalikan hasil ke skala asli
    forecast = np.exp(forecast_log)

else:
    # Fit ulang ETS ke seluruh data latih
    tuned_ets_model = ExponentialSmoothing(train['Total_Revenue'], trend='add', seasonal='add', seasonal_periods=7).fit()

    # Prediksi 30 hari ke depan
    forecast = tuned_ets_model.forecast(steps=steps_ahead)

# Siapkan tanggal untuk hasil prediksi
forecast.index = pd.date_range(start='2020-01-01', periods=steps_ahead)

# Jika kamu memiliki actual test data 30 hari juga (misal Maret), evaluasi bisa dilakukan
# Kalau tidak, langkah evaluasi di bawah hanya ilustratif
# Ubah test ke test_30 jika tersedia, atau sesuaikan:
test_30 = test['Total_Revenue'][:steps_ahead]  # Ambil 30 hari pertama dari test untuk evaluasi

# Hitung metrik evaluasi
mape = mean_absolute_percentage_error(test_30, forecast)
mae = mean_absolute_error(test_30, forecast)
rmse = np.sqrt(mean_squared_error(test_30, forecast))

# Tampilkan hasil evaluasi
print(f"MAPE: {mape:.2%}")
print(f"MAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")

MAPE: 9.29%
MAE : 7,831.43
RMSE: 9,314.34


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


In [ ]:
forecast_januari = forecast

forecast_januari_df = forecast_januari.to_frame().reset_index()

# Ganti nama kolom agar lebih deskriptif dan rapi
forecast_januari_df.columns = ['Date', 'Forecasted_Revenue']

# Tampilkan beberapa baris teratas sebagai preview
forecast_januari_df


,Date,Forecasted_Revenue
0,2020-01-01,85577.689937
1,2020-01-02,85366.685690
2,2020-01-03,85390.606466
3,2020-01-04,85387.891348
4,2020-01-05,85388.199484
5,2020-01-06,85388.164514
6,2020-01-07,85388.168482
7,2020-01-08,85388.168032
8,2020-01-09,85388.168083
9,2020-01-10,85388.168077


# Model Untuk Prediksi Transaksi

In [ ]:
# Buat model ARIMA untuk data time series
# Di sini kita gunakan log(train) agar model lebih stabil terhadap variasi besar (log transform)
# Parameter (1,1,1) artinya: p=1 (Auto REGRESSION), d=1 (differencing), q=1 (MOVING AVERAGE)

model_arima = ARIMA(np.log(train['Quantity Ordered']), order=(1,1,1))  # Buat model ARIMA dengan log-transformed data
result_arima = model_arima.fit()  # Latih model ARIMA

# Forecast dalam bentuk log selama panjang data uji (misal Maret = 31 hari)
forecast_arima_log = result_arima.forecast(steps=len(test))

# Kembalikan hasil forecast dari skala log ke skala asli dengan fungsi eksponensial
forecast_arima = np.exp(forecast_arima_log)  # Ini adalah hasil prediksi revenue harian

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


In [ ]:
# Buat model ETS (Exponential Smoothing) dengan komponen tren dan musiman
# 'trend="add"' artinya kita mengasumsikan adanya tren naik/turun yang bersifat aditif
# 'seasonal="add"' artinya ada pola musiman (berulang) yang juga ditambahkan
# 'seasonal_periods=7' menunjukkan adanya pola mingguan (7 hari)

model_ets = ExponentialSmoothing(train['Quantity Ordered'], trend='add', seasonal='add', seasonal_periods=7)  # Buat model ETS
result_ets = model_ets.fit()  # Latih modelnya

# Lakukan prediksi sebanyak panjang data uji
forecast_ets = result_ets.forecast(steps=len(test))  # Hasil prediksi revenue harian dari model ETS

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


In [ ]:
# Hitung MAPE (Mean Absolute Percentage Error) untuk mengukur akurasi kedua model
# Semakin kecil nilai MAPE, semakin akurat prediksi model terhadap data aktual

mape_arima = mean_absolute_percentage_error(test['Quantity Ordered'], forecast_arima)  # Error ARIMA
mape_ets = mean_absolute_percentage_error(test['Quantity Ordered'], forecast_ets)      # Error ETS

# Tampilkan hasil evaluasi dalam format persentase
print(f'MAPE ARIMA: {mape_arima:.2%}')  # Misalnya: 8.45%
print(f'MAPE ETS: {mape_ets:.2%}')

MAPE ARIMA: 28.63%
MAPE ETS: 27.31%


In [ ]:
# Bandingkan nilai MAPE dari ARIMA dan ETS
# Pilih model dengan MAPE terkecil sebagai model terbaik

best_model = "ARIMA" if mape_arima < mape_ets else "ETS"  # Jika MAPE ARIMA lebih kecil, pilih ARIMA, jika tidak pilih ETS

# Tampilkan model terbaik berdasarkan evaluasi MAPE
print(f"Model terbaik berdasarkan MAPE adalah: {best_model}")

Model terbaik berdasarkan MAPE adalah: ETS


In [ ]:
# Forecast ulang selama 30 hari (1 bulan ke depan) menggunakan model terbaik yang sudah ditentukan
steps_ahead = 30  # Prediksi 30 hari ke depan

if best_model == "ARIMA":
    # Fit ulang ARIMA ke seluruh data latih
    tuned_arima_model = ARIMA(np.log(train['Quantity Ordered']), order=(1,1,1)).fit()

    # Prediksi 30 hari ke depan dalam skala log
    forecast_log = tuned_arima_model.forecast(steps=steps_ahead)

    # Kembalikan hasil ke skala asli
    forecast = np.exp(forecast_log)

else:
    # Fit ulang ETS ke seluruh data latih
    tuned_ets_model = ExponentialSmoothing(train['Quantity Ordered'], trend='add', seasonal='add', seasonal_periods=7).fit()

    # Prediksi 30 hari ke depan
    forecast = tuned_ets_model.forecast(steps=steps_ahead)

# Siapkan tanggal untuk hasil prediksi
forecast.index = pd.date_range(start='2020-01-01', periods=steps_ahead)

# Jika kamu memiliki actual test data 30 hari juga (misal Maret), evaluasi bisa dilakukan
# Kalau tidak, langkah evaluasi di bawah hanya ilustratif
# Ubah test ke test_30 jika tersedia, atau sesuaikan:
test_30 = test['Quantity Ordered'][:steps_ahead]  # Ambil 30 hari pertama dari test untuk evaluasi

# Hitung metrik evaluasi
mape = mean_absolute_percentage_error(test_30, forecast)
mae = mean_absolute_error(test_30, forecast)
rmse = np.sqrt(mean_squared_error(test_30, forecast))

# Tampilkan hasil evaluasi
print(f"MAPE: {mape:.2%}")
print(f"MAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


MAPE: 4.51%
MAE : 22.79
RMSE: 30.97


In [ ]:
forecast_januari = forecast

forecast_januari_df = forecast_januari.to_frame().reset_index()

# Ganti nama kolom agar lebih deskriptif dan rapi
forecast_januari_df.columns = ['Date', 'Quantity Ordered']

# Tampilkan beberapa baris teratas sebagai preview
forecast_januari_df

,Date,Quantity Ordered
0,2020-01-01,503.317995
1,2020-01-02,514.160388
2,2020-01-03,519.680641
3,2020-01-04,508.639849
4,2020-01-05,508.859655
5,2020-01-06,519.894947
6,2020-01-07,516.148455
7,2020-01-08,509.827380
8,2020-01-09,520.669773
9,2020-01-10,526.190027


# Pertanyaan yang Perlu Dijawab:



1.   Hitunglah total revenue, jumlah order, dan jumlah barang yang terjual sepanjang tahun 2019. Selain itu, hitung rata-rata jumlah barang yang dibeli per transaksi dan rata-rata spending per transaksi.




In [ ]:
df_clean['Time'] = df_clean['Order Date'].dt.time
df_clean['Order Date'] = df_clean['Order Date'].dt.date
df_clean

/tmp/ipython-input-2447426471.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Time'] = df_clean['Order Date'].dt.time
/tmp/ipython-input-2447426471.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Order Date'] = df_clean['Order Date'].dt.date


,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Revenue,Time
0,141234,iPhone,1,700.00,2019-01-22,"944 Walnut St, Boston, MA 02215",700.00,21:25:00
1,141235,Lightning Charging Cable,1,14.95,2019-01-28,"185 Maple St, Portland, OR 97035",14.95,14:15:00
2,141236,Wired Headphones,2,11.99,2019-01-17,"538 Adams St, San Francisco, CA 94016",23.98,13:33:00
3,141237,27in FHD Monitor,1,149.99,2019-01-05,"738 10th St, Los Angeles, CA 90001",149.99,20:33:00
4,141238,Wired Headphones,1,11.99,2019-01-25,"387 10th St, Austin, TX 73301",11.99,11:59:00
...,...,...,...,...,...,...,...,...
186845,319666,Lightning Charging Cable,1,14.95,2019-12-11,"14 Madison St, San Francisco, CA 94016",14.95,20:58:00
186846,319667,AA Batteries (4-pack),2,3.84,2019-12-01,"549 Willow St, Los Angeles, CA 90001",7.68,12:01:00
186847,319668,Vareebadd Phone,1,400.00,2019-12-09,"273 Wilson St, Seattle, WA 98101",400.00,06:43:00
186848,319669,Wired Headphones,1,11.99,2019-12-03,"778 River St, Dallas, TX 75001",11.99,10:39:00


In [ ]:
# Jika Anda ingin sum dan mean untuk kedua kolom tersebut
df_a = df_clean.groupby('Product').agg({
    'Revenue': ['sum', 'mean'],
    'Quantity Ordered': ['sum', 'mean']
}).reset_index()

# Opsional: Meratakan nama kolom agar tidak bertingkat (multi-index)
df_a.columns = ['Product', 'Revenue_Sum', 'Revenue_Mean', 'Quantity_Sum', 'Quantity_Mean']
df_a

,Product,Revenue_Sum,Revenue_Mean,Quantity_Sum,Quantity_Mean
0,20in Monitor,453818.74,110.741518,4126,1.006833
1,27in 4K Gaming Monitor,2433147.61,390.867086,6239,1.002249
2,27in FHD Monitor,1131074.59,150.850172,7541,1.005735
3,34in Ultrawide Monitor,2352898.08,381.097843,6192,1.002915
4,AA Batteries (4-pack),105937.92,5.157138,27588,1.343005
5,AAA Batteries (4-pack),92537.51,4.493639,30949,1.502889
6,Apple Airpods Headphones,2345550.00,151.082126,15637,1.007214
7,Bose SoundSport Headphones,1342865.70,100.982531,13430,1.009926
8,Flatscreen TV,1443900.00,301.188986,4813,1.003963
9,Google Phone,3317400.00,600.760594,5529,1.001268




2.   Hitunglah jumlah order dan GMV yang diperoleh dengan rentang waktu berikut:

*   Harian

*   Mingguan

*   Bulanan









In [ ]:
df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'])
df_timed = df_clean.set_index('Order Date')
monthly_analysis = df_timed.resample('MS').agg({
    'Order ID': 'count',
    'Revenue': 'sum'
}).reset_index()
monthly_analysis

/tmp/ipython-input-1418747556.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'])


,Order Date,Order ID,Revenue
0,2019-01-01,9665,1812742.87
1,2019-02-01,11953,2200012.30
2,2019-03-01,15125,2804954.57
3,2019-04-01,18254,3389203.47
4,2019-05-01,16544,3150537.62
5,2019-06-01,13533,2576265.21
6,2019-07-01,14273,2646434.43
7,2019-08-01,11938,2241042.83
8,2019-09-01,11602,2094453.70
9,2019-10-01,20243,3734714.66


In [ ]:
daily_analysis = df_timed.resample('D').agg({
    'Order ID': 'count',
    'Revenue': 'sum'
}).reset_index()
daily_analysis

,Order Date,Order ID,Revenue
0,2019-01-01,302,65681.94
1,2019-01-02,323,70663.20
2,2019-01-03,296,47046.20
3,2019-01-04,293,62000.22
4,2019-01-05,308,46524.63
...,...,...,...
361,2019-12-28,816,133601.53
362,2019-12-29,839,156005.83
363,2019-12-30,807,151857.82
364,2019-12-31,763,131439.32


In [ ]:
weekly_analysis = df_timed.resample('W').agg({
    'Order ID': 'count',
    'Revenue': 'sum'
}).reset_index()
weekly_analysis

,Order Date,Order ID,Revenue
0,2019-01-06,1812,344678.73
1,2019-01-13,2196,409389.43
2,2019-01-20,2198,394921.11
3,2019-01-27,2235,426020.17
4,2019-02-03,2473,459570.62
5,2019-02-10,2960,565300.05
6,2019-02-17,3052,568166.39
7,2019-02-24,2984,535177.61
8,2019-03-03,3178,578303.45
9,2019-03-10,3430,640496.38


3. Tim marketing ingin mengetahui produk apa saja yang paling sering dibeli dalam 1 tahun terakhir. Rencananya, mereka akan mencoba mem-bundling top produk ini untuk meningkatkan penjualan. Identifikasi top 10 produk yang membawa revenue terbesar dalam 3 bulan terakhir dan produk apa saja yang bisa di-bundling berdasarkan hasil analisis.

In [ ]:
# 1. Pastikan Order Date sudah tipe datetime
df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'])

# 2. Cari tanggal terakhir di dataset dan hitung batas 3 bulan ke belakang
last_date = df_clean['Order Date'].max()
three_months_ago = last_date - pd.DateOffset(months=3)

# 3. Filter data 3 bulan terakhir
df_last_3m = df_clean[df_clean['Order Date'] >= three_months_ago]

# 4. Hitung Top 10 Produk berdasarkan Revenue
top_10_products = df_last_3m.groupby('Product')['Revenue'].sum().nlargest(10).reset_index()

top_10_products

/tmp/ipython-input-623472378.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'])


,Product,Revenue
0,Macbook Pro Laptop,2735300.00
1,iPhone,1602300.00
2,ThinkPad Laptop,1373986.26
3,Google Phone,1083000.00
4,27in 4K Gaming Monitor,842378.40
5,Apple Airpods Headphones,788400.00
6,34in Ultrawide Monitor,786579.30
7,Flatscreen TV,494100.00
8,Bose SoundSport Headphones,453354.66
9,27in FHD Monitor,370775.28


4.  identifikasi top 5 kota yang memiliki order terbanyak dan 5 kota yang memiliki total dan rata-rata spending terbesar.

In [ ]:
df_clean['City'] = df_clean['Purchase Address'].apply(lambda x: x.split(',')[1].strip())
top_5_percentage = df_clean['City'].value_counts(normalize=True).head(5)*100
top_5_percentage

/tmp/ipython-input-3580846411.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['City'] = df_clean['Purchase Address'].apply(lambda x: x.split(',')[1].strip())


,proportion
City,
San Francisco,24.052597
Los Angeles,15.923378
New York City,13.380809
Boston,10.717037
Atlanta,8.005322


5.  Tim marketing ingin mengetahui kapan penjualan mencapai titik tertinggi sehingga mereka bisa merancang strategi marketing. Analisis pada rentang jam berapa penjualan terjadi secara aktif (rush hour).

In [ ]:
import datetime

def waktu(Time):
  if Time >= datetime.time(5, 0, 0) and Time <= datetime.time(10, 59, 0):
    return 'Pagi'
  elif Time >= datetime.time(11, 0, 0) and Time <= datetime.time(14, 59, 0):
    return 'Siang'
  elif Time >= datetime.time(15, 0, 0) and Time <= datetime.time(18, 0, 0):
    return 'Sore'
  elif Time >= datetime.time(19, 0, 0) and Time <= datetime.time(23, 59, 0):
    return 'Malam'
  else:
    return 'Dini Hari'

In [ ]:
df_clean['Kondisi']=df_clean['Time'].apply(waktu)
df_clean

/tmp/ipython-input-3546891398.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Kondisi']=df_clean['Time'].apply(waktu)


,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Revenue,Time,City,Kondisi
0,141234,iPhone,1,700.00,2019-01-22,"944 Walnut St, Boston, MA 02215",700.00,21:25:00,Boston,Malam
1,141235,Lightning Charging Cable,1,14.95,2019-01-28,"185 Maple St, Portland, OR 97035",14.95,14:15:00,Portland,Siang
2,141236,Wired Headphones,2,11.99,2019-01-17,"538 Adams St, San Francisco, CA 94016",23.98,13:33:00,San Francisco,Siang
3,141237,27in FHD Monitor,1,149.99,2019-01-05,"738 10th St, Los Angeles, CA 90001",149.99,20:33:00,Los Angeles,Malam
4,141238,Wired Headphones,1,11.99,2019-01-25,"387 10th St, Austin, TX 73301",11.99,11:59:00,Austin,Siang
...,...,...,...,...,...,...,...,...,...,...
186845,319666,Lightning Charging Cable,1,14.95,2019-12-11,"14 Madison St, San Francisco, CA 94016",14.95,20:58:00,San Francisco,Malam
186846,319667,AA Batteries (4-pack),2,3.84,2019-12-01,"549 Willow St, Los Angeles, CA 90001",7.68,12:01:00,Los Angeles,Siang
186847,319668,Vareebadd Phone,1,400.00,2019-12-09,"273 Wilson St, Seattle, WA 98101",400.00,06:43:00,Seattle,Pagi
186848,319669,Wired Headphones,1,11.99,2019-12-03,"778 River St, Dallas, TX 75001",11.99,10:39:00,Dallas,Pagi


In [ ]:
df_clean['Kondisi'].value_counts()

,count
Kondisi,
Malam,51068
Siang,48034
Pagi,33711
Sore,31579
Dini Hari,21247


# Kesimpulan :

Kesimpulanya ialah pada :
1.  pertanyaan pertama  rata-rata jumlah barang yang dibeli per transaksi dan rata-rata spending per transaksi. di dapat Total Transaksi di dominasi oleh pembelian Type C namun secara pendapatan iphone lebih unggul daripada produk yang lain.
2.  Pertanyaan Kedua yang paling tinggi secara pendapatan ialah pada pembelian di bulan desember hal tersebut pembelian ada event musiman yaitu event nataru natal dan tahun baru
3.  Pertanyaan Ketiga produk paling laris ialah macbook pro dan kedua iphone secara pendapatanya
4.  Pertanyaan Ke empat Kota yang paling banyak peminat pembelinya ialah kota San Frasisco
5.  Pertanyaan ke lima orang orang melakukan transaksi di toko ini kebanyakan di malam hari di jam 18:00 - 00:00.
6.  Pertanyaan Ke enam setelah di lakukan forecasting dengan menggunakan 2 uji arima dengan ets di dapat prediksi dalam 1 bulan kedepan bulan januari 2026 dari sini kita bisa mengetahui prediksi pendapatan dan predikssi kebutuhan barangnya

# Rekomendasi

1. Untuk Trend Penjualan Meningkatkan Penjualan Sebaiknya di lakukan product campaign di malam hari
2. Menaikan Minat Atau Penjualan Secara Drastis antar daeerah sebaiknya di lakukan mini event dengan penjualan terbanyak dan melakukan puket bundling di darah yang di minati
3. Melakukn promosi agresif pada saat event nataru karena peminat lebih banyak daripada bulan seperti biasanya